# اختبار «الشراء بعد أسابيع الهبوط» — السوق السعودي

**من الجوال مباشرة، بدون تثبيت أي شيء.** اضغط ▶️ على كل خلية بالترتيب (أو من القائمة: Runtime ← Run all).

- الخلية 1: الإعدادات — عدّل السهم والفترة ثم أعد التشغيل.
- الخلية 2: تحميل الملفات (مرة واحدة).
- الخلية 3: تشغيل الاختبار.
- الخلية 4: رسم الصفقات.

> النتائج تعليمية وليست توصية استثمارية.


In [ ]:
#@title 1) الإعدادات { display-mode: "form" }
SYMBOL = "1120.SR"   # 1120=الراجحي | 1180=الأهلي | 1010=الرياض | 2010=سابك | 2222=أرامكو | 7010=stc
START = "2026-01-01"  # بداية المتاجرة الفعلية
HISTORY_START = "2025-01-01"  # تاريخ أقدم لمتوسط 200 يوم (اتركه)
N_DOWN = 3            # عدد أسابيع الهبوط المتتالية المطلوبة
HOLD_WEEKS = 4        # مدة الاحتفاظ بالأسابيع
STOP_PCT = 7.0        # وقف الخسارة %
FEE_BPS = 15.5        # العمولة لكل جهة (نقطة أساس)
COMPARE = True        # مقارنة الفلاتر (اتجاه السوق / حجم التداول)
SWEEP = False         # مسح شبكي لكل المعاملات


In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/aqaqpop1999-max/nautilus_trader.git"
BRANCH = "arena/c765ce04-nautilus-trader"
WORKDIR = "/content/nautilus_trader"

if not os.path.isdir(WORKDIR + "/analysis"):
    subprocess.run(
        ["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO, WORKDIR],
        check=True,
    )

TOOL_DIR = WORKDIR + "/analysis/saudi_market"
os.chdir(TOOL_DIR)
print("جاهز:", os.getcwd())


In [ ]:
cmd = [
    sys.executable, "run_backtest.py",
    "--yahoo", SYMBOL,
    "--start", START,
    "--history-start", HISTORY_START,
    "--symbol", SYMBOL.split(".")[0],
    "--n-down", str(N_DOWN),
    "--hold-weeks", str(HOLD_WEEKS),
    "--stop", str(STOP_PCT),
    "--fee-bps", str(FEE_BPS),
    "--mode", "vector",
    "--export-trades", "trades.csv",
]
if COMPARE:
    cmd.append("--compare")
if SWEEP:
    cmd.append("--sweep")

print(" ".join(cmd), "\n")
import subprocess
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


In [ ]:
import pandas as pd
from IPython.display import display

try:
    df = pd.read_csv("trades.csv")
except FileNotFoundError:
    df = pd.DataFrame()

if df.empty:
    print("لا توجد صفقات في هذه الفترة — جرّب --start أبكر أو N_DOWN أصغر.")
else:
    df["exit_week"] = pd.to_datetime(df["exit_week"])
    df["equity"] = (1 + df["return"]).cumprod()
    display(df)

    df.plot(x="exit_week", y="equity", marker="o", legend=False,
            title="منحنى رأس المال (1.0 = رأس المال الابتدائي)")
    df.plot.bar(x="entry_week", y="return", legend=False, title="عائد كل صفقة")
    import matplotlib.pyplot as plt
    plt.axhline(0, color="black", linewidth=0.8)
    plt.show()


## كيف تقرأ النتيجة

1. **عدد الصفقات** — أقل من 10 صفقات = لا تستنتج شيئًا.
2. **أقصى تراجع** — هل تتحمل خسارة مؤقتة بهذا الحجم؟
3. **العائد التراكمي مقابل «الشراء والاحتفاظ»** — أقل منه = الفكرة لا تستحق.
4. **جدول المقارنة** — هل فلتر الاتجاه/الحجم حسّن النتيجة أم قلّل الصفقات فقط؟

### أفكار للتجربة
- غيّر `SYMBOL` إلى `2010.SR` (سابك) أو `1010.SR` (بنك الرياض).
- اجعل `START = "2024-01-01"` لبيانات أطول (أدق إحصائيًا).
- اجعل `COMPARE = True` و`SWEEP = True` معًا.
- جرّب `STOP_PCT = 4` أو `HOLD_WEEKS = 2`.

### تنبيهات
- بيانات Yahoo قد تحتوي فجوات، ولا تتوفر لمؤشر تاسي نفسه (`^TASI.SR`).
- لا تُنمذج التسوية T+2 ولا حدود التذبذب ±10% ولا توزيعات الأرباح.
- 2026 وحدها عيّنة صغيرة (≈ 44 أسبوعًا)؛ النتائج الإيجابية قد تكون حظًا.
- ليست توصية استثمارية.
